# YOLOv8n Baseline Evaluation, Dataset V1
> Notebook note: This notebook was reorganised after model training. It uses the existing model weights and does not repeat the 30-epoch training.
## Overview
This notebook evaluates the V1 baseline on the cleaned test set.
## Method
The dataset is cleaned using `dataset_preparation.py`. Model performance is measured using precision, recall, mAP50 and mAP50–95.
## Files
*   Dataset: `dataset/pork_rasher_v1.zip`
*   Weights: `model_runs/yolov8n_v1_baseline/weights/best.pt`
*   Results: `model_runs/yolov8n_v1_baseline_test*`
## Limitation
The 80-image test split contains only seven twisted-meat annotations, limiting conclusions about that class.

In [2]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from pathlib import Path
import sys

# Dataset and model files on Google Drive.
PROJECT_ROOT = Path("/content/drive/MyDrive/MMA3001_Project")
ZIP_PATH = PROJECT_ROOT / "dataset" / "pork_rasher_v1.zip"
CLEAN_DIR = Path("/content/pork_rasher_dataset_v1_clean")
MANIFEST_PATH = PROJECT_ROOT / "dataset" / "manifests" / "v1_preparation.json"
RUNS_DIR = PROJECT_ROOT / "model_runs"

# Shared dataset preparation code.
module_folders = [PROJECT_ROOT / "src", Path.cwd() / "src", Path("/content/src"), Path("/content")]
module_dir = next((p for p in module_folders if (p / "dataset_preparation.py").is_file()), None)
if module_dir is None:
    raise FileNotFoundError("dataset_preparation.py not found in the project src folder")
sys.path.insert(0, str(module_dir))
from dataset_preparation import prepare_dataset, print_audit, assert_preparations_match

In [4]:
# Remove overlapping source images from Dataset V1.
manifest = prepare_dataset(
    zip_path=ZIP_PATH,
    output_dir=CLEAN_DIR,
    expected_dataset_version=1,
    require_labels=True,
)
print("Removed overlaps:", manifest["removed_overlap_images"])
print_audit(manifest["audit"])

# Verify the cleaned split sizes and class names.
expected_images = {"train": 1935, "valid": 119, "test": 80}
expected_classes = ["loose-meat", "packaging-error", "twisted-meat", "unsealed"]
image_counts = {s: x["images"] for s, x in manifest["audit"]["splits"].items()}
assert image_counts == expected_images, (image_counts, expected_images)
assert manifest["audit"]["class_names"] == expected_classes
assert manifest["moved_to_validation"] == 0
clean = CLEAN_DIR
print("Dataset V1 checked.")


Removed overlaps: {'train': 9, 'valid': 1}

TRAIN: 1935 images; 29 backgrounds; 0 missing label files
  loose-meat: 139 boxes
  packaging-error: 931 boxes
  twisted-meat: 91 boxes
  unsealed: 1107 boxes

VALID: 119 images; 0 backgrounds; 0 missing label files
  loose-meat: 5 boxes
  packaging-error: 57 boxes
  twisted-meat: 5 boxes
  unsealed: 87 boxes

TEST: 80 images; 1 backgrounds; 0 missing label files
  loose-meat: 3 boxes
  packaging-error: 37 boxes
  twisted-meat: 7 boxes
  unsealed: 52 boxes
Dataset V1 checked.


In [5]:
# Verify the dataset preparation record, if available.
if MANIFEST_PATH.exists():
    assert_preparations_match(manifest, MANIFEST_PATH)
    print("Preparation record matches:", MANIFEST_PATH)
else:
    print("No existing preparation record; split counts and classes were checked.")

Preparation record matches: /content/drive/MyDrive/MMA3001_Project/dataset/manifests/v1_preparation.json


In [6]:
%pip install -q ultralytics
from ultralytics import YOLO

# Load the saved V1 baseline model.
model_path = RUNS_DIR / "yolov8n_v1_baseline" / "weights" / "best.pt"
if not model_path.exists():
    raise FileNotFoundError(f"Baseline weights not found: {model_path}")
model = YOLO(str(model_path))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.0/47.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.6/97.6 kB 10.8 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [7]:
# Evaluate on the cleaned test split.
test_results = model.val(
    data=str(clean / "data.yaml"),
    split="test",
    imgsz=640,
    batch=16,
    device=0,
    plots=True,
    project=str(RUNS_DIR),
    name="yolov8n_v1_baseline_test",
)
print("Test evaluation complete.")

Ultralytics 8.4.183 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1653.9±633.5 MB/s, size: 73.1 KB)
val: Scanning /content/pork_rasher_dataset_v1_clean/test/labels... 80 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 80/80 1.2Kit/s 0.1s
val: New cache created: /content/pork_rasher_dataset_v1_clean/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 1.4it/s 3.5s
                   all         80         99      0.721      0.518      0.582      0.241
            loose-meat          2          3       0.83      0.667      0.775      0.234
       packaging-error         26         37      0.582      0.351      0.436      0.124
          twisted-meat          7          7      0.727      0.286      0.363      0.182
              unsealed         51         52     

## Recorded Test Results
The YOLOv8n baseline achieved **72.1% precision**, **51.8% recall**, **58.2% mAP50** and **24.1% mAP50–95** on 80 test images with 99 annotations.